# 03 - Park effects by archetype

**Method:** on balls in play, `woba_gap = actual wOBA - expected wOBA`. Expected wOBA comes from exit velocity and launch angle and does not know the park, so the gap picks up park effects (plus defense, weather, and luck). For each park and archetype we compare the gap to the league-wide gap for the same archetype, then shrink small samples toward zero.

`pitcher_edge` is positive when a park helps that archetype.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
from src.config import PROCESSED, INTERIM, FIGURES
from src.park_effects import contact_frame, park_archetype_effects, best_archetypes
from src.plots import plot_park_archetypes

df = pd.read_parquet(INTERIM / "pitches_archetyped.parquet")
contact = contact_frame(df)
print(f"{len(contact):,} balls in play")

In [ ]:
effects = park_archetype_effects(contact, by_hand=True, shrink_k=200)
effects.to_csv(PROCESSED / "park_archetype_effects.csv", index=False)
effects.head()

## Check cell sizes

Each row of `effects` is one park/archetype/handedness cell. Cells with few balls in play are noisy (they are shrunk toward zero, and the charts hide cells under 100). If most cells are small, use fewer archetype tiers or more seasons.

In [ ]:
print(effects["bip"].describe().round(0))
print(f"Cells with 100+ balls in play: {(effects['bip'] >= 100).mean():.0%}")

## Sanity check: park-level ranking

Average gap by park, in wOBA points. Compare this ranking to the park factors leaderboard on Baseball Savant. It should line up broadly (Coors Field near the top, for example). If it looks wrong, fix the data or park labels before going further.

In [ ]:
(contact.groupby("park")["woba_gap"].mean() * 1000).sort_values(ascending=False).round(1)

## Robustness: split-half stability

Compute effects separately on the earlier and later seasons. If the effects are real, they should be positively correlated. Report this number on your methodology slide, whatever it turns out to be.

In [ ]:
seasons = sorted(contact["game_year"].unique())
half = len(seasons) // 2
a = park_archetype_effects(contact[contact["game_year"].isin(seasons[:half])])
b = park_archetype_effects(contact[contact["game_year"].isin(seasons[half:])])
m = a.merge(b, on=["park", "archetype", "p_throws"], suffixes=("_a", "_b"))
m = m[(m["bip_a"] >= 100) & (m["bip_b"] >= 100)]
print(f"{len(m)} park/archetype cells, correlation: {m['effect_raw_a'].corr(m['effect_raw_b']):.2f}")

## Example park

In [ ]:
best_archetypes(effects, "Coors Field", hand="R")

In [ ]:
plot_park_archetypes(effects, "Coors Field", "R", path=FIGURES / "Coors_Field_RHP.png")
plt.show()

## Make a chart for every park

In [ ]:
for park in sorted(effects["park"].unique()):
    for hand in ["R", "L"]:
        fig = plot_park_archetypes(effects, park, hand, path=FIGURES / f"{park.replace(' ', '_')}_{hand}HP.png")
        if fig is not None:
            plt.close(fig)

## Commit

```
git add notebooks/03_park_effects.ipynb data/processed/park_archetype_effects.csv slides/figures
git commit -m "park effects by archetype with stability check"
git push
```